# CAR — scale runs (7B and 14B) on an A100 80GB

Runtime → Change runtime type → **A100 GPU**, High-RAM on. Run the cells top to bottom.
Every result file is written to Google Drive (`MyDrive/CAR_scale/`) after **each layer**, so a disconnect loses at most one layer;
re-running a cell restarts that model from scratch, so after a disconnect skip the cells whose JSON is already complete
(the log shows `Saved ...`).

Priority order: **1** (Bias in Bios audit, the headline) → **2** (StereoSet + factorial conditional entanglement) → **3** (steering).
Bring back every `.json` in `CAR_scale/` plus `log.txt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/CAR_scale'; os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
print('results ->', OUT)

In [ ]:
!pip -q install -U transformers accelerate datasets scikit-learn

In [ ]:
# code: upload CAR_colab.zip when prompted (only needed once per runtime)
import os, zipfile
os.makedirs('/content/car', exist_ok=True)
if not os.path.exists('/content/car/car_audit2.py'):
    from google.colab import files
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('/content/car')
print(sorted(os.listdir('/content/car')))

## 1. Bias in Bios audit (headline): Qwen2.5-7B, Qwen2.5-14B — 6 evenly spaced layers, 3 seeds

In [ ]:
%cd /content/car
!python -u car_audit2.py --models Qwen/Qwen2.5-7B --half --bs 32 --datasets bios --layers frac6 --seeds 3 --out $OUT/car_audit2_G_bios_7b.json 2>&1 | tee -a $OUT/log.txt

In [ ]:
%cd /content/car
!python -u car_audit2.py --models Qwen/Qwen2.5-14B --half --bs 32 --datasets bios --layers frac6 --seeds 3 --out $OUT/car_audit2_G_bios_14b.json 2>&1 | tee -a $OUT/log.txt

## 2. StereoSet + factorial (conditional entanglement, LEACE, Σ⁻¹ filter)

In [ ]:
%cd /content/car
!python -u car_audit2.py --models Qwen/Qwen2.5-7B --half --bs 32 --datasets stereoset factorial --template-cells --layers frac6 --seeds 3 --out $OUT/car_audit2_G_7b.json 2>&1 | tee -a $OUT/log.txt

In [ ]:
%cd /content/car
!python -u car_audit2.py --models Qwen/Qwen2.5-14B --half --bs 32 --datasets stereoset factorial --template-cells --layers frac6 --seeds 3 --out $OUT/car_audit2_G_14b.json 2>&1 | tee -a $OUT/log.txt

## 3. Steering (readouts after steering, equal-norm controls)

In [ ]:
%cd /content/car
!python -u car_steer.py --models Qwen/Qwen2.5-7B Qwen/Qwen2.5-14B --half --out $OUT/car_steer_v3_big.json 2>&1 | tee -a $OUT/log.txt

## Optional: a second family (Llama-3.1-8B, gated)
Accept the license at huggingface.co/meta-llama/Llama-3.1-8B first, then paste a read token when asked.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
%cd /content/car
!python -u car_audit2.py --models meta-llama/Llama-3.1-8B --half --bs 32 --datasets bios --layers frac6 --seeds 3 --out $OUT/car_audit2_G_bios_llama8b.json 2>&1 | tee -a $OUT/log.txt